# PM100D Power Measurement
Single power measurement using Thorlabs PM100D + S144C integrating sphere.

In [ ]:
print("hello")  # for initial test

hello


In [1]:
import pyvisa
import numpy as np
import os
import time
import matplotlib.pyplot as plt

In [2]:
# Test to find visa address
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print(resources)

('ASRL8::INSTR', 'ASRL9::INSTR', 'ASRL12::INSTR', 'ASRL13::INSTR')


In [4]:
# Configuration

# Filename (Change manually for each run)
filename = "blow50"

# Folder to save data
save_folder = r"C:\Users\leach\OneDrive - Danmarks Tekniske Universitet\4. semester\Fagprojekt\Kode til Lab data\experimentData" # add another folder to seperate types

# Measurement wavelength in nm (set to the same as the laser)
wavelength_nm = 1550

# PM100D VISA address
visa_address = "USB0::0x1313::0x8078::P0006823::INSTR"

# Hardware averaging count (1 to 3000)
hw_averages = 50

# Number of software-side repeat readings to average
sw_repeats = 300

# Check if save folder exists, build file path
os.makedirs(save_folder, exist_ok=True)
file_path = os.path.join(save_folder, filename + ".txt")

# Quick safety check to prevent accidental manual overwrites
if os.path.exists(file_path):
    print(f"WARNING: {file_path} already exists!")
    print("To avoid losing data, change the 'filename' variable at the top and re-run.")
    raise FileExistsError("Execution stopped to prevent overwriting existing data.")

# Connect to PM100D
rm = pyvisa.ResourceManager()
pm = rm.open_resource(visa_address)

print(f"Connected to: {pm.query('*IDN?').strip()}")

# Configure PM100D for maximum precision
pm.write("CONF:POW")                           # Power measurement mode
pm.write(f"CORR:WAV {wavelength_nm}")          # Set calibration wavelength
pm.write(f"SENS:AVER:COUN {hw_averages}")      # Hardware averaging
pm.write("SENS:POW:RANG:AUTO ON")              # Auto-range
pm.write("INP:FILT:LPAS:STAT ON")              # Low-pass filter on

# Verify settings
print(f"Wavelength:        {pm.query('CORR:WAV?').strip()} nm")
print(f"Averaging count:   {pm.query('SENS:AVER:COUN?').strip()}")
print(f"Auto-range:        {pm.query('SENS:POW:RANG:AUTO?').strip()}")
print(f"Low-pass filter:   {pm.query('INP:FILT:LPAS:STAT?').strip()}")

# Take sw_repeats readings and average
readings_W = np.empty(sw_repeats)
timeStamps_W = np.empty(sw_repeats)
startTime = time.time()

for i in range(sw_repeats):
    readings_W[i] = float(pm.query("MEAS:POW?"))
    timeReading = time.time()
    timeStamps_W[i] = timeReading - startTime
    print(f"  Reading {i+1}/{sw_repeats}: {readings_W[i]*1e3:.6f} mW, Time {timeStamps_W[i]:.2f} s")
    time.sleep(0.01) # should maybe be removed, but might give stability

# Find dBm
power_mW = np.mean(readings_W) * 1e3
if power_mW <= 0:
    raise ValueError(f"Measured power is {power_mW:.6f} mW — cannot convert to dBm. Check optical input.")
power_dBm = 10 * np.log10(power_mW)

print(f"\nResult: {power_mW:.6f} mW  /  {power_dBm:.4f} dBm")
print(f"Std of {sw_repeats} readings: {np.std(readings_W)*1e3:.6f} mW")


# Save in txt format
with open(file_path, "w") as f:
    f.write("Power (mW)\tTime (sek)\n")  # Header
    for i in range(sw_repeats):
        f.write(f"{readings_W[i]*1e3:.6f}\t{timeStamps_W[i]:.4f}\n") # Converts W to mW automatically on save to match header
        
    # Add final statistics at the bottom
    f.write(f"\nMean Power (dBm):\t{power_dBm:.4f}\n")
    f.write(f"Mean Power (mW):\t{power_mW:.6f}\n")

print(f"\n>>> Data successfully saved to: {file_path} <<<")


# Plot to visualize as sanity check, in case something went wrong and the file should be deleted.
# Passes time array as X, and mW array as Y
plt.plot(timeStamps_W, readings_W * 1e3, color='b') 
plt.xlabel("Time [seconds]")
plt.ylabel("Wattage [mW]")
plt.title(f"Undisturbed fiber - {filename}")
plt.grid(True)
plt.show()

# Close connection
pm.close()
rm.close()
print("Connection closed.")

VisaIOError: VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.